[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 Easy: Cross-Entropy Loss

Implement **cross-entropy loss** from scratch.

$$\text{CE}(x, y) = -\log\frac{e^{x_y}}{\sum_j e^{x_j}}$$

### Signature
```python
def cross_entropy_loss(logits: Tensor, targets: Tensor) -> Tensor:
    # logits: (B, C) float, targets: (B,) long indices
    # Returns: scalar loss (mean over batch)
```

### Rules
- Do NOT use `F.cross_entropy` or `nn.CrossEntropyLoss`
- Must be numerically stable (use logsumexp trick)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


In [2]:
import torch

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

"""
logits          (B, C)
log_sum_exp     (B, 1)   ← logsumexp, keepdim=True
log_probs       (B, C)   ← broadcasting으로 뺄셈
correct_log_probs (B,)   ← 정답 위치만 인덱싱
loss            scalar   ← mean()
"""

def cross_entropy_loss(logits, targets):
    # 첫번째 차원의 크기. 즉 샘플의 개수를 가져옴
    B = targets.shape[0]

    # log(sum(e^x))을 조건대로 stable하게 계산
    # shape는 (B, 1)이 됨
    # dim=-1이라 마지막 차원(클래스 방향)으로 합산. -> c개 클래스를 합침
    # (dim=0 하면 B개를 합치는게 됨..?!)
    log_sum_exp = torch.logsumexp(logits, dim=-1, keepdim=True)

    # log 확률 계산
    # broadcasting돼서 (B, 1) -> (B, C)
    # 각 행마다 그 행의 logsumexp를 빼줌
    log_probs = logits - log_sum_exp

    # 정답 클래스의 log 확률만 뽑기
    # log_probs는 (B, C)인데 지금 각 샘플의 정답 클래스 딱 하나씩만 필요
    # 그래서 다시 (B, )로 만듦
    correct_log_probs = log_probs[torch.arange(B), targets]

    # 평균내고 부호 -로
    loss = -correct_log_probs.mean()

    return loss

In [4]:
# 🧪 Debug
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('Loss:', cross_entropy_loss(logits, targets))
print('Ref: ', torch.nn.functional.cross_entropy(logits, targets))

Loss: tensor(3.2416)
Ref:  tensor(3.2416)


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('cross_entropy')


🧪 Testing: Cross-Entropy Loss (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Matches F.cross_entropy (6.6ms)
  ✅ [2/4] Numerical stability (1.0ms)
  ✅ [3/4] Scalar output (0.3ms)
  ✅ [4/4] Gradient flow (19.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (27.1ms total)
  Progress saved. Run status() to see your dashboard.

